# BYVA Phase 3 — Colab T4 training run

Runs the real Phase 3 gate on a Colab NVIDIA T4 (16 GB): BPE 32k tokenizer
training, then a SMALL-config pretraining run (D-0013) with FP16 and gradient
accumulation (D-0018) on Phase 2 shards.

Before starting, read `docs/PHASES.md` (gate definition), `docs/PHASE3_REPORT.md`
(status), and `docs/DECISIONS.md` (D-0013..D-0018). Record only results that
actually happen in this session.

In [ ]:
REPO_BRANCH = "byva/implementation"
!git clone --depth 1 -b {REPO_BRANCH} https://github.com/val-alpha/byva-v1.git /content/byva-v1
%cd /content/byva-v1
!git log --oneline -1

In [ ]:
import torch, subprocess
print(torch.__version__, torch.cuda.is_available())
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"
!nvidia-smi

## 1. Data

Upload verified JSONL (`{"text": "..."}` per line, e.g. c4 `id` + FineWeb `en`
slices per `docs/DATA.md`) to `/content/data/raw.jsonl`, or mount Drive and set
`RAW_JSONL`. Record the id:en mixture weight (D-0016) in `MIXTURE_NOTE`.

In [ ]:
RAW_JSONL = "/content/data/raw.jsonl"
MIXTURE_NOTE = "id:en = ?  (fill in per D-0016)"
SHARD_DIR = "/content/shards"
!python3 training/build_shards.py --input {RAW_JSONL} --output-dir {SHARD_DIR}
!cat {SHARD_DIR}/manifest.json | head -40

## 2. Train the 32,768 BPE table (prerequisite, D-0015)

In [ ]:
ARTIFACT = "/content/byva-tokenizer-32k-v1.json"
!python3 training/train_bpe.py --shards {SHARD_DIR} --target-vocab 32768 --version 1 --output {ARTIFACT} --stats /content/bpe-stats.json
!cat /content/bpe-stats.json

## 3. SMALL training run on the T4 (FP16 + gradient accumulation)

In [ ]:
CKPT = "/content/drive/MyDrive/byva/ckpt-small-t4.pt"
from google.colab import drive
drive.mount('/content/drive')
!python3 training/train.py --shards {SHARD_DIR} --config small --tokenizer {ARTIFACT} --device cuda --fp16 --grad-accum 4 --steps 10000 --seq-len 1024 --batch-size 8 --checkpoint {CKPT}

### Resume after a disconnected session (D-0014)

In [ ]:
!python3 training/train.py --shards {SHARD_DIR} --config small --tokenizer {ARTIFACT} --device cuda --fp16 --grad-accum 4 --steps 10000 --seq-len 1024 --batch-size 8 --checkpoint {CKPT} --resume

## 4. Record the gate evidence

In [ ]:
import torch, dataclasses, sys
sys.path.insert(0, "training")
from train import load_checkpoint
model, optimizer, state, scaler = load_checkpoint(CKPT)
print("steps:", state.step)
print("loss:", state.loss_history[0], "->", state.loss_history[-1])
print("config:", dataclasses.asdict(model.config))
print("mixture:", MIXTURE_NOTE)

Copy the printed evidence into `docs/PHASE3_REPORT.md` (loss curve, val loss,
generation sample) and the agent progress note. The Phase 3 gate passes only if
loss actually fell on held-out data with the trained BPE tokenizer.